# SelvaSonic ML — Grad-CAM: Visualización de activaciones sobre espectrogramas

**Autores:** Laura Ruiz Arango · Jose Aldair Molina Méndez  
**Asignatura:** Aprendizaje Automático  
**Profesor:** Alcides Montoya  
**Fecha:** Junio 2026

---

## Propósito

El notebook 13 demostró que **Attention v2** supera al baseline en todas las métricas
globales. Pero los números no explican *por qué*. Este notebook responde esa pregunta
visualmente: ¿qué regiones del espectrograma (tiempo × frecuencia) usa cada modelo
para tomar su decisión?

Usamos **Grad-CAM** (Gradient-weighted Class Activation Mapping, Selvaraju et al. 2017):
los gradientes del logit de la clase objetivo respecto a las activaciones de la última
capa convolucional se usan como pesos para construir un mapa de calor espacial.

| Modelo | Checkpoint |
|---|---|
| `SelvaSonicCNN` (baseline) | `baseline_S3_v2_20260527_0118/best.pth` |
| `SelvaSonicCNNAttention` v2 | `attention_S4_v2_20260602_1332/best.pth` |

**Capa hookeada:** `model.feature_extractor[-1]` — el 4.° bloque conv (Conv2d 128→256 +
BN + ReLU + MaxPool). Activaciones de forma **(B, 256, 8, ~13)** antes del pooling global.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, NUM_CLASSES, N_MELS
from src.dataset import create_dataloaders
from src.model import SelvaSonicCNN, SelvaSonicCNNAttention

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Rutas a checkpoints ───────────────────────────────────────────────────
CKPT_BASELINE = PROJECT_ROOT / 'results/runs/baseline_S3_v2_20260527_0118/best.pth'
CKPT_V2       = PROJECT_ROOT / 'results/runs/attention_S4_v2_20260602_1332/best.pth'
OUT_DIR       = PROJECT_ROOT / 'results/gradcam'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for nombre, path in [('baseline', CKPT_BASELINE), ('attention_v2', CKPT_V2)]:
    if not path.exists():
        raise FileNotFoundError(f'Checkpoint faltante para {nombre}: {path}')
    print(f'  {nombre}: OK')

# Archivos por clase (para títulos informativos)
ARCHIVOS_POR_CLASE: dict[str, int] = {
    'no_ave': 1600, 'Celeus_grammicus': 28, 'Chordeiles_pusillus': 21,
    'Crypturellus_cinereus': 48, 'Crypturellus_undulatus': 29,
    'Frederickena_fulva': 20, 'Glaucidium_brasilianum': 22,
    'Lipaugus_vociferans': 36, 'Ramphastos_tucanus': 31,
    'Rupornis_magnirostris': 20, 'Trogon_viridis': 79,
}

## Carga del test DataLoader y los 2 modelos

Los modelos se cargan en `eval()` mode — BatchNorm usa estadísticas fijas —
pero **sin** `torch.no_grad()`, porque Grad-CAM necesita que el grafo de
gradientes esté habilitado durante el forward y el backward.

In [ ]:
_, _, test_loader, label_map = create_dataloaders(
    DATA_DIR / 'raw',
    batch_size=32,
    num_workers=0,
    random_state=42,
    verbose=False,
)

CLASES: list[str] = [nombre for nombre, _ in sorted(label_map.items(), key=lambda x: x[1])]
IDX_A_CLASE: dict[int, str] = {v: k for k, v in label_map.items()}
print(f'Test set: {len(test_loader.dataset)} clips | Clases: {CLASES}\n')


def _cargar_modelo(
    *,
    arquitectura: type,
    checkpoint_path: Path,
    device: torch.device,
) -> nn.Module:
    """Instancia la arquitectura y carga los pesos del checkpoint."""
    ckpt   = torch.load(str(checkpoint_path), map_location=device, weights_only=False)
    modelo = arquitectura(num_classes=NUM_CLASSES)
    modelo.load_state_dict(ckpt['model_state_dict'])
    modelo.to(device).eval()
    acc = ckpt.get('best_val_acc', 0)
    print(f'  {checkpoint_path.parent.name} | epoch={ckpt.get("epoch","?")} | best_val_acc={acc:.4f}')
    return modelo


print('Cargando modelos...')
modelo_baseline = _cargar_modelo(
    arquitectura=SelvaSonicCNN,
    checkpoint_path=CKPT_BASELINE,
    device=DEVICE,
)
modelo_v2 = _cargar_modelo(
    arquitectura=SelvaSonicCNNAttention,
    checkpoint_path=CKPT_V2,
    device=DEVICE,
)

# Verificar que ambos tienen feature_extractor[-1] (el bloque que hookearemos)
for nombre, modelo in [('baseline', modelo_baseline), ('v2', modelo_v2)]:
    capa = modelo.feature_extractor[-1]
    print(f'  feature_extractor[-1] de {nombre}: {type(capa).__name__}')

## Implementación de Grad-CAM

Algoritmo (Selvaraju et al., ICCV 2017):
1. Forward pass → activaciones A^k de la capa objetivo guardadas via hook.
2. Backward del logit de la clase objetivo → gradientes ∂y^c/∂A^k via hook.
3. Pesos α_k = GlobalAveragePool(∂y^c/∂A^k) — importancia de cada canal.
4. CAM = ReLU(Σ_k α_k · A^k) — combinación lineal positiva de feature maps.
5. Upsample bilinear al tamaño del input original.
6. Normalizar a [0, 1].

In [ ]:
class GradCAM:
    """Grad-CAM con hooks de PyTorch. Funciona con cualquier bloque conv nn.Module."""

    def __init__(self, modelo: nn.Module, capa_objetivo: nn.Module) -> None:
        self.modelo = modelo
        self.activaciones: torch.Tensor | None = None
        self.gradientes: torch.Tensor | None = None
        self._handles = [
            capa_objetivo.register_forward_hook(self._hook_activaciones),
            capa_objetivo.register_full_backward_hook(self._hook_gradientes),
        ]

    def _hook_activaciones(
        self,
        modulo: nn.Module,
        entrada: tuple,
        salida: torch.Tensor,
    ) -> None:
        # Guardamos la salida del bloque (B, 256, H', T') para usarla en el CAM
        self.activaciones = salida.detach()

    def _hook_gradientes(
        self,
        modulo: nn.Module,
        grad_entrada: tuple,
        grad_salida: tuple,
    ) -> None:
        # grad_salida[0] = ∂Loss/∂(output del bloque) = lo que queremos para Grad-CAM
        self.gradientes = grad_salida[0].detach()

    def calcular(
        self,
        x: torch.Tensor,
        clase_objetivo: int,
    ) -> np.ndarray:
        """Forward + backward y construye el CAM normalizado.

        Args:
            x: Tensor de espectrograma (1, 1, N_MELS, T) en el device del modelo.
            clase_objetivo: Índice de la clase cuyo logit usamos para el backward.

        Returns:
            CAM normalizado a [0, 1], shape (N_MELS, T).
        """
        self.modelo.zero_grad()
        salida = self.modelo(x)                         # logits (1, NUM_CLASSES)
        salida[0, clase_objetivo].backward()            # gradientes fluyen hasta hooks

        # α_k = promedio espacial de los gradientes por canal
        pesos = self.gradientes.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)

        # CAM = ReLU(suma ponderada de feature maps)
        cam = (pesos * self.activaciones).sum(dim=1, keepdim=True)  # (1, 1, H', T')
        cam = F.relu(cam)

        # Upsample al tamaño del espectrograma de entrada
        cam = F.interpolate(
            cam, size=x.shape[2:], mode='bilinear', align_corners=False
        )                                               # (1, 1, N_MELS, T)

        cam_np = cam.squeeze().cpu().numpy()            # (N_MELS, T)
        if cam_np.max() > 0:
            cam_np = cam_np / cam_np.max()
        return cam_np

    def limpiar(self) -> None:
        """Elimina los hooks para liberar memoria."""
        for h in self._handles:
            h.remove()


print('Clase GradCAM definida.')
print(f'  Capa a hookear: feature_extractor[-1] (Conv2d 128\u219256, BN, ReLU, MaxPool)')

## Funciones auxiliares

In [ ]:
def plot_gradcam_comparativo(
    *,
    espectro: np.ndarray,
    cam_baseline: np.ndarray,
    cam_v2: np.ndarray,
    nombre_clase: str,
    confianza_baseline: float,
    confianza_v2: float,
    pred_correcta: bool,
    pred_nombre_b: str,
    pred_nombre_v2: str,
    ax_row: np.ndarray,
) -> None:
    """Plotea una fila de 3 columnas: espectro original, CAM baseline, CAM v2."""
    icono = '\u2713' if pred_correcta else '\u2717'

    # Col 0: espectrograma original en escala de color
    ax_row[0].imshow(espectro, aspect='auto', origin='lower', cmap='magma')
    ax_row[0].set_title(f'Espectrograma\n[GT: {nombre_clase}]', fontsize=8)
    ax_row[0].set_xlabel('Frame temporal', fontsize=7)
    ax_row[0].set_ylabel('Banda Mel', fontsize=7)

    # Col 1: espectrograma + CAM baseline
    ax_row[1].imshow(espectro, aspect='auto', origin='lower', cmap='gray')
    im1 = ax_row[1].imshow(
        cam_baseline, aspect='auto', origin='lower',
        cmap='jet', alpha=0.55, vmin=0, vmax=1,
    )
    estado_b = '\u2713 correcto' if pred_nombre_b == nombre_clase else f'\u2717 -> {pred_nombre_b[:14]}'
    ax_row[1].set_title(
        f'Baseline ({estado_b})\nconf={confianza_baseline:.2f}', fontsize=8,
    )
    ax_row[1].set_xlabel('Frame temporal', fontsize=7)
    plt.colorbar(im1, ax=ax_row[1], fraction=0.046, pad=0.04)

    # Col 2: espectrograma + CAM v2
    ax_row[2].imshow(espectro, aspect='auto', origin='lower', cmap='gray')
    im2 = ax_row[2].imshow(
        cam_v2, aspect='auto', origin='lower',
        cmap='jet', alpha=0.55, vmin=0, vmax=1,
    )
    estado_v2 = '\u2713 correcto' if pred_nombre_v2 == nombre_clase else f'\u2717 -> {pred_nombre_v2[:14]}'
    ax_row[2].set_title(
        f'Attention v2 ({estado_v2})\nconf={confianza_v2:.2f}', fontsize=8,
    )
    ax_row[2].set_xlabel('Frame temporal', fontsize=7)
    plt.colorbar(im2, ax=ax_row[2], fraction=0.046, pad=0.04)


def calcular_entropia_espacial(cam: np.ndarray) -> float:
    """Entropía normalizada de la distribución espacial del CAM.

    Entropía baja ≈ foco concentrado en pocas regiones.
    Entropía alta ≈ activación dispersa por todo el espectrograma.
    """
    cam_flat = cam.flatten()
    total    = cam_flat.sum()
    if total == 0:
        return 0.0
    p    = cam_flat / total
    eps  = 1e-10
    h    = -np.sum(p * np.log2(p + eps))
    hmax = np.log2(len(p))           # máxima entropía posible para este tamaño
    return float(h / hmax) if hmax > 0 else 0.0


def calcular_centroide_frecuencia(cam: np.ndarray) -> float:
    """Centroide de frecuencia del CAM: banda Mel media ponderada por activación.

    Centroide alto (>64) = el modelo mira frecuencias altas.
    Centroide bajo (<64) = el modelo mira frecuencias bajas.
    """
    freq_profile = cam.mean(axis=1)   # (N_MELS,) — promedio sobre el eje temporal
    total        = freq_profile.sum()
    if total == 0:
        return float(cam.shape[0] / 2)
    freq_indices = np.arange(cam.shape[0], dtype=float)
    return float(np.dot(freq_indices, freq_profile) / total)


print('Helpers definidos.')

## Selección de muestras del test set

Para cada una de las 11 clases buscamos hasta **2 muestras** donde ambos modelos
predicen correctamente. Para las muestras de error buscamos casos donde **solo
baseline falla** pero v2 acierta — los más informativos para entender las diferencias.

La selección se hace en `torch.no_grad()` (solo necesitamos predicciones, no gradientes).
Los tensores de las muestras seleccionadas se almacenan en CPU para su uso posterior.

In [ ]:
MAX_POR_CLASE = 2
MAX_ERRORES   = 6

# muestras_correctas: clase_idx -> lista de (x_cpu, y, pred_b, conf_b, pred_v2, conf_v2)
muestras_correctas: dict[int, list] = {i: [] for i in range(NUM_CLASSES)}
muestras_error: list = []      # baseline falla, v2 acierta

print('Iterando test set para seleccionar muestras...')
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_dev    = x_batch.to(DEVICE)
        probs_b  = F.softmax(modelo_baseline(x_dev), dim=1).cpu()
        probs_v2 = F.softmax(modelo_v2(x_dev), dim=1).cpu()

        for i in range(x_batch.size(0)):
            y       = int(y_batch[i])
            pred_b  = int(probs_b[i].argmax())
            conf_b  = float(probs_b[i].max())
            pred_v2 = int(probs_v2[i].argmax())
            conf_v2 = float(probs_v2[i].max())
            x_cpu   = x_batch[i:i+1].cpu()  # (1, 1, N_MELS, T) en CPU

            # Ambos correctos: prioridad a alta confianza media
            if pred_b == y and pred_v2 == y:
                if len(muestras_correctas[y]) < MAX_POR_CLASE:
                    muestras_correctas[y].append(
                        (x_cpu, y, pred_b, conf_b, pred_v2, conf_v2)
                    )

            # Baseline falla, v2 acierta
            if pred_b != y and pred_v2 == y and len(muestras_error) < MAX_ERRORES:
                muestras_error.append(
                    (x_cpu, y, pred_b, conf_b, pred_v2, conf_v2)
                )

# Resumen de selección
total_correctas = sum(len(v) for v in muestras_correctas.values())
print(f'\nMuestras donde ambos aciertan: {total_correctas}')
for idx, clase in enumerate(CLASES):
    n = len(muestras_correctas[idx])
    flag = '' if n == MAX_POR_CLASE else f'  <- solo {n}'
    print(f'  {clase:<30} {n}{flag}')
print(f'\nMuestras de error (baseline falla, v2 acierta): {len(muestras_error)}')

---
## Sección A — Grad-CAM en predicciones correctas

Para las muestras donde **ambos modelos aciertan**, la pregunta es:
¿miran al mismo lugar del espectrograma, o desarrollaron focos diferentes?

Si v2 tiene activaciones más concentradas en la banda frecuencial característica
de la especie (e.g., el trino de alta frecuencia de Chordeiles), mientras baseline
activa de forma más difusa, eso explicaría por qué v2 generaliza mejor:
aprendió features más específicas del canto, no simplemente el ruido de fondo.

Cada figura muestra **una clase** con hasta 2 muestras (2 filas × 3 columnas).
Rojo-amarillo en el heatmap = máxima activación; azul = baja activación.

In [ ]:
# Crear GradCAM una vez y reusar en secciones A, B y C
gc_base = GradCAM(modelo_baseline, modelo_baseline.feature_extractor[-1])
gc_v2   = GradCAM(modelo_v2,       modelo_v2.feature_extractor[-1])

archivos_sA: list[str] = []

for clase_idx, clase_nombre in enumerate(CLASES):
    muestras = muestras_correctas.get(clase_idx, [])
    if not muestras:
        print(f'  {clase_nombre}: sin muestras correctas, se omite')
        continue

    n_filas = len(muestras)
    fig, axes = plt.subplots(n_filas, 3, figsize=(15, n_filas * 4.5))
    fig.patch.set_facecolor('#F8F8F8')
    if n_filas == 1:
        axes = axes[np.newaxis, :]

    for fila, (x_cpu, y_true, pred_b, conf_b, pred_v2, conf_v2) in enumerate(muestras):
        x_dev  = x_cpu.to(DEVICE)
        # GradCAM usa la clase verdadera (que ambos predicen correctamente)
        cam_b  = gc_base.calcular(x_dev, y_true)
        cam_v2_arr = gc_v2.calcular(x_dev, y_true)
        spec   = x_cpu.squeeze().numpy()      # (N_MELS, T)

        plot_gradcam_comparativo(
            espectro=spec,
            cam_baseline=cam_b,
            cam_v2=cam_v2_arr,
            nombre_clase=clase_nombre,
            confianza_baseline=conf_b,
            confianza_v2=conf_v2,
            pred_correcta=True,
            pred_nombre_b=IDX_A_CLASE[pred_b],
            pred_nombre_v2=IDX_A_CLASE[pred_v2],
            ax_row=axes[fila],
        )

    n_arch = ARCHIVOS_POR_CLASE.get(clase_nombre, '?')
    plt.suptitle(
        f'Grad-CAM | {clase_nombre} ({n_arch} arch. entrenamiento)',
        fontsize=11, y=1.01,
    )
    plt.tight_layout()
    fname = OUT_DIR / f'correctas_{clase_nombre}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight', facecolor='#F8F8F8')
    plt.show()
    archivos_sA.append(str(fname))
    print(f'  Guardado: {fname.name}')

print(f'\nSección A: {len(archivos_sA)} figuras guardadas en {OUT_DIR}')

### Interpretación

Observa cada par de CAMs (baseline vs v2) y busca estos patrones:

- **Foco concentrado vs disperso**: ¿las activaciones de v2 son más precisas
  (un parche rojo bien definido) mientras baseline activa difusamente?
- **Banda de frecuencia**: ¿el heatmap cae en la región característica de la especie?
  Para aves con cantos agudos (Chordeiles, Glaucidium), esperamos activación en
  bandas Mel altas (parte superior del espectrograma).
- **Coherencia temporal**: ¿las activaciones siguen la estructura temporal del canto
  (sílabas, trinos) o son aleatorias en el tiempo?
- **Clase no_ave**: baseline y v2 deberían mirar regiones distintas a las de aves;
  la activación puede ser más uniforme (el modelo aprendió 'ausencia de estructura').

---
## Sección B — Grad-CAM en errores del baseline (v2 acierta)

Estos son los casos más reveladores: **baseline predice mal, v2 predice bien**.
Si el Grad-CAM muestra que baseline mira a la región equivocada del espectrograma
(baja frecuencia cuando la especie canta en alta frecuencia, o el ruido de fondo
en vez del canto), tenemos evidencia visual directa de por qué falla.

Los títulos reflejan las predicciones de cada modelo para facilitar la comparación.

In [ ]:
if not muestras_error:
    print('No se encontraron muestras de error (baseline falla, v2 acierta).')
    print('Puede ocurrir si ambos modelos tienen patrones de error similares.')
else:
    n_err  = len(muestras_error)
    fig, axes = plt.subplots(n_err, 3, figsize=(15, n_err * 4.5))
    fig.patch.set_facecolor('#F8F8F8')
    if n_err == 1:
        axes = axes[np.newaxis, :]

    for fila, (x_cpu, y_true, pred_b, conf_b, pred_v2, conf_v2) in enumerate(muestras_error):
        x_dev      = x_cpu.to(DEVICE)
        # Para baseline: clase que PREDIJO (incorrecta) — ver qué activó ese error
        # Para v2:       clase que PREDIJO (correcta)
        cam_b      = gc_base.calcular(x_dev, pred_b)
        cam_v2_arr = gc_v2.calcular(x_dev, pred_v2)
        spec       = x_cpu.squeeze().numpy()
        clase_gt   = IDX_A_CLASE[y_true]

        plot_gradcam_comparativo(
            espectro=spec,
            cam_baseline=cam_b,
            cam_v2=cam_v2_arr,
            nombre_clase=clase_gt,
            confianza_baseline=conf_b,
            confianza_v2=conf_v2,
            pred_correcta=False,
            pred_nombre_b=IDX_A_CLASE[pred_b],
            pred_nombre_v2=IDX_A_CLASE[pred_v2],
            ax_row=axes[fila],
        )

    plt.suptitle(
        'Grad-CAM — Errores del baseline (v2 acierta)\n'
        'Col 1: activación del logit que baseline PREDIJO (incorrecto) | '
        'Col 2: activación de la clase CORRECTA en v2',
        fontsize=10, y=1.01,
    )
    plt.tight_layout()
    fname = OUT_DIR / 'errores_baseline.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight', facecolor='#F8F8F8')
    plt.show()
    print(f'Guardado: {fname}')

### Interpretación

Para cada error, compara las dos CAMs con la pregunta:
**¿Dónde miró baseline para predecir la clase incorrecta? ¿Dónde miró v2 para acertar?**

Hipótesis típicas:
- Baseline activa en zonas de ruido ambiental (baja frecuencia, < banda 30) mientras
  v2 activa en la banda del canto de la especie (frecuencia media-alta).
- Baseline confunde dos especies similares (e.g., los dos Crypturellus) porque activa
  en features genéricas; v2 discrimina por un feature específico de la especie correcta.
- Si las CAMs son visualmente similares pero v2 tiene mayor confianza en la clase
  correcta, la ventaja puede venir del módulo de attention, no de la localización.

---
## Sección C — Análisis cuantitativo de las CAMs

Las inspecciones visuales son subjetivas. Cuantificamos dos propiedades de cada CAM:

**1. Entropía espacial normalizada** — ¿qué tan concentrado está el foco?
- Entropía 0 = toda la activación en un único píxel (máximo foco).
- Entropía 1 = activación uniforme en todo el espectrograma (sin foco).
- Si v2 tiene menor entropía que baseline: el attention ayudó a concentrar
  la atención en la región relevante del espectrograma.

**2. Centroide de frecuencia** — ¿en qué banda Mel se concentra la activación?
- Rango: 0 (baja frecuencia, ~50 Hz) a 127 (alta frecuencia, ~11 kHz).
- Si v2 tiene centroide más alto para especies de canto agudo: aprendió
  a mirar la región de frecuencia correcta.

In [ ]:
# Acumular métricas para todas las muestras correctas
entropias_b:   list[float] = []
entropias_v2:  list[float] = []
centroides_b:  list[float] = []
centroides_v2: list[float] = []

# Por clase (para el scatter)
cent_clase_b:  dict[str, list[float]] = {c: [] for c in CLASES}
cent_clase_v2: dict[str, list[float]] = {c: [] for c in CLASES}
entr_clase_b:  dict[str, list[float]] = {c: [] for c in CLASES}
entr_clase_v2: dict[str, list[float]] = {c: [] for c in CLASES}

print('Calculando métricas cuantitativas sobre muestras correctas...')
for clase_idx, clase_nombre in enumerate(CLASES):
    for (x_cpu, y_true, pred_b, conf_b, pred_v2, conf_v2) in muestras_correctas.get(clase_idx, []):
        x_dev      = x_cpu.to(DEVICE)
        cam_b_arr  = gc_base.calcular(x_dev, y_true)
        cam_v2_arr = gc_v2.calcular(x_dev, y_true)

        e_b  = calcular_entropia_espacial(cam_b_arr)
        e_v2 = calcular_entropia_espacial(cam_v2_arr)
        c_b  = calcular_centroide_frecuencia(cam_b_arr)
        c_v2 = calcular_centroide_frecuencia(cam_v2_arr)

        entropias_b.append(e_b);   entropias_v2.append(e_v2)
        centroides_b.append(c_b);  centroides_v2.append(c_v2)
        cent_clase_b[clase_nombre].append(c_b)
        cent_clase_v2[clase_nombre].append(c_v2)
        entr_clase_b[clase_nombre].append(e_b)
        entr_clase_v2[clase_nombre].append(e_v2)

# Liberar hooks — no se usan más
gc_base.limpiar()
gc_v2.limpiar()

print(f'\nMuestras analizadas: {len(entropias_b)}')
print(f'Entropia media   — baseline: {np.mean(entropias_b):.3f} | v2: {np.mean(entropias_v2):.3f}')
print(f'Centroide medio  — baseline: {np.mean(centroides_b):.1f}  | v2: {np.mean(centroides_v2):.1f}')

# ── Plot 1: histogramas de entropía ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#F8F8F8')

ax = axes[0]
ax.hist(entropias_b,  bins=12, range=(0,1), color='#FD79A8', alpha=0.65,
        edgecolor='#FD79A8', label=f'Baseline  (media={np.mean(entropias_b):.3f})', density=True)
ax.hist(entropias_v2, bins=12, range=(0,1), color='#00B894', alpha=0.65,
        edgecolor='#00B894', label=f'Attention v2  (media={np.mean(entropias_v2):.3f})', density=True)
ax.set_xlabel('Entropía espacial normalizada', fontsize=10)
ax.set_ylabel('Densidad', fontsize=10)
ax.set_title(
    'Distribución de entropía de las CAMs\n'
    'Menor entropía = foco más concentrado',
    fontsize=10,
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Plot 2: centroide de frecuencia por clase ─────────────────────────────
ax = axes[1]
clases_con_datos = [c for c in CLASES if cent_clase_b[c] or cent_clase_v2[c]]
x_pos = np.arange(len(clases_con_datos))

med_b  = [np.mean(cent_clase_b[c])  if cent_clase_b[c]  else np.nan for c in clases_con_datos]
med_v2 = [np.mean(cent_clase_v2[c]) if cent_clase_v2[c] else np.nan for c in clases_con_datos]

ax.scatter(x_pos - 0.15, med_b,  s=90, c='#FD79A8', zorder=3,
           edgecolors='#2D3436', linewidths=0.6, label='Baseline')
ax.scatter(x_pos + 0.15, med_v2, s=90, c='#00B894', zorder=3,
           edgecolors='#2D3436', linewidths=0.6, label='Attention v2')
for xp, b, v in zip(x_pos, med_b, med_v2):
    if not (np.isnan(b) or np.isnan(v)):
        ax.plot([xp - 0.15, xp + 0.15], [b, v], '-', color='gray', alpha=0.4, lw=1)

ax.axhline(N_MELS / 2, linestyle='--', color='gray', lw=1, alpha=0.5, label='Frecuencia media (banda 64)')
ax.set_xticks(x_pos)
ax.set_xticklabels(
    [c.replace('_', '\n')[:18] for c in clases_con_datos],
    fontsize=6.5, rotation=30, ha='right',
)
ax.set_ylabel('Centroide de frecuencia (banda Mel)', fontsize=10)
ax.set_title(
    'Centroide de frecuencia por clase\n'
    'Arriba de 64 = modelo mira frecuencias altas',
    fontsize=10,
)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fname_c = OUT_DIR / 'analisis_cuantitativo.png'
plt.savefig(fname_c, dpi=120, bbox_inches='tight', facecolor='#F8F8F8')
plt.show()
print(f'Guardado: {fname_c}')

### Interpretación

**Histograma de entropía:**
- Si la distribución de v2 (verde) está desplazada a la **izquierda** de baseline (rosa),
  el modelo v2 tiene un foco más concentrado — el attention lo dirige hacia regiones
  específicas del espectrograma en lugar de distribuirse uniformemente.
- Si las distribuciones se solapan completamente, la ganancia de v2 viene de
  otro mecanismo (class weights, mejor representación, etc.).

**Scatter de centroide de frecuencia:**
- Para clases con cantos agudos (Chordeiles, Glaucidium, Frederickena), esperamos
  centroides altos (> 80, frecuencias > 6 kHz).
- Si v2 tiene centroides más coherentes con la bioacústica de la especie
  (más altos para aves agudas, más bajos para aves graves), confirma que el
  módulo de attention aprendió a focalizar en la banda de frecuencia relevante.

---
## Conclusión

### Diferencias cualitativas observadas

Tras inspeccionar las Secciones A y B, anota aquí tus observaciones clave:

- ¿Las CAMs de v2 son más concentradas o más difusas que las del baseline?
- ¿Hay clases donde ambos modelos miran el mismo lugar, y clases donde difieren mucho?
- En los errores del baseline (Sección B), ¿se aprecia visualmente que miró al lugar
  incorrecto (ruido ambiental, frecuencia equivocada)?

### Hallazgos cuantitativos

| Métrica | Baseline | Attention v2 | Interpretación |
|---|---|---|---|
| Entropía media | *(ver output Sec C)* | *(ver output Sec C)* | Menor = más concentrado |
| Centroide Mel medio | *(ver output Sec C)* | *(ver output Sec C)* | > 64 = frecuencias altas |

### Limitaciones del análisis

1. **Tamaño de muestra pequeño**: con 2 muestras por clase, las conclusiones estadísticas
   son débiles. El análisis cuantitativo de la Sección C es orientativo.
2. **Grad-CAM en atención**: hookear `feature_extractor[-1]` en v2 captura las activaciones
   **antes** del bloque de attention. El CAM muestra qué features CNN causan la predicción,
   pero no visualiza directamente los patrones de atención (para eso se necesitaría
   visualizar los attention weights de `self.attention`).
3. **Resolución espacial del CAM**: la salida de `feature_extractor[-1]` tiene resolución
   (8, ~13) antes del upsample, lo que limita la precisión del mapa de calor.
4. **Normalización del espectrograma**: el input al modelo está normalizado (z-score),
   por lo que los colores del espectrograma en el fondo no corresponden directamente
   a la energía acústica (son valores normalizados, no dB).

### Conexión con el notebook 13

El notebook 13 mostró que v2 tiene menor ECE (mejor calibración) y mayor macro F1.
Este notebook añade la dimensión visual: si las CAMs de v2 muestran activaciones más
coherentes con la bioacústica de cada especie, la mejora no es solo estadística —
el modelo aprendió representaciones más interpretables y específicas del dominio.

In [ ]:
resumen_gradcam: dict = {
    'descripcion': 'Resumen cuantitativo del análisis Grad-CAM (NB 14)',
    'fecha': '2026-06-02',
    'n_muestras_correctas': len(entropias_b),
    'n_muestras_error': len(muestras_error),
    'entropia': {
        'baseline_media':  float(np.mean(entropias_b))  if entropias_b  else None,
        'baseline_std':    float(np.std(entropias_b))   if entropias_b  else None,
        'v2_media':        float(np.mean(entropias_v2)) if entropias_v2 else None,
        'v2_std':          float(np.std(entropias_v2))  if entropias_v2 else None,
    },
    'centroide_frecuencia': {
        'baseline_media':  float(np.mean(centroides_b))  if centroides_b  else None,
        'baseline_std':    float(np.std(centroides_b))   if centroides_b  else None,
        'v2_media':        float(np.mean(centroides_v2)) if centroides_v2 else None,
        'v2_std':          float(np.std(centroides_v2))  if centroides_v2 else None,
    },
    'por_clase': {
        clase: {
            'entropia_baseline_media':  float(np.mean(entr_clase_b[clase]))  if entr_clase_b[clase]  else None,
            'entropia_v2_media':        float(np.mean(entr_clase_v2[clase])) if entr_clase_v2[clase] else None,
            'centroide_baseline_media': float(np.mean(cent_clase_b[clase]))  if cent_clase_b[clase]  else None,
            'centroide_v2_media':       float(np.mean(cent_clase_v2[clase])) if cent_clase_v2[clase] else None,
        }
        for clase in CLASES
    },
    'capa_hookeada': 'feature_extractor[-1]',
    'resolucion_cam_antes_upsample': '(B, 256, 8, ~13)',
}

salida_json = OUT_DIR / 'resumen.json'
with open(salida_json, 'w', encoding='utf-8') as f:
    json.dump(resumen_gradcam, f, indent=2, ensure_ascii=False)

print(f'Guardado: {salida_json}')
print(f'\nArtefactos en {OUT_DIR}:')
for p in sorted(OUT_DIR.glob('*.png')):
    print(f'  {p.name}')
print(f'  resumen.json')